# Language Translator ML Project (TensorFlow Version)
## Multi-language translation system supporting 10 languages

**Languages Supported:**
- English (en_US)
- German (de_DE)
- Hindi (hi_IN)
- Spanish (es_ES)
- French (fr_FR)
- Italian (it_IT)
- Arabic (ar_SA)
- Dutch (nl_NL)
- Japanese (ja_JP)
- Portuguese (pt_PT)

## Step 1: Import Libraries and Load Dataset

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import os
from collections import Counter
warnings.filterwarnings('ignore')

# TensorFlow imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# Set plotting style
plt.style.use('ggplot')
sns.set_palette("husl")
print("✓ Libraries imported successfully")
print(f"✓ TensorFlow version: {tf.__version__}")

In [ ]:
from datasets import load_dataset

ds = load_dataset("Amani27/massive_translation_dataset")
ds

## Step 2: Exploratory Data Analysis (EDA)

In [ ]:
# Convert dataset to pandas DataFrame for easier analysis
train_df = pd.DataFrame(ds['train'])
val_df = pd.DataFrame(ds['validation'])
test_df = pd.DataFrame(ds['test'])

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Total samples: {len(train_df) + len(val_df) + len(test_df)}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nShape: {train_df.shape}")
print("\n" + "=" * 60)
print("FIRST 5 SAMPLES")
print("=" * 60)
train_df.head()

In [ ]:
# Check for missing values
print("=" * 60)
print("MISSING VALUES CHECK")
print("=" * 60)
missing_data = train_df.isnull().sum()
print(missing_data)
print(f"\nTotal missing values: {missing_data.sum()}")

# Check data types
print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)
print(train_df.dtypes)

# Basic statistics
print("\n" + "=" * 60)
print("BASIC INFORMATION")
print("=" * 60)
train_df.info()

In [ ]:
# Analyze text lengths across languages
languages = ['en_US', 'de_DE', 'hi_IN', 'es_ES', 'fr_FR', 'it_IT', 'ar_SA', 'nl_NL', 'ja_JP', 'pt_PT']

print("=" * 60)
print("TEXT LENGTH STATISTICS (in characters)")
print("=" * 60)

length_stats = {}
for lang in languages:
    lengths = train_df[lang].str.len()
    length_stats[lang] = {
        'mean': lengths.mean(),
        'median': lengths.median(),
        'min': lengths.min(),
        'max': lengths.max(),
        'std': lengths.std()
    }
    
length_df = pd.DataFrame(length_stats).T
print(length_df.round(2))

## Step 3: Data Visualization

In [ ]:
# Visualize dataset split distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dataset split sizes
splits = ['Train', 'Validation', 'Test']
sizes = [len(train_df), len(val_df), len(test_df)]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

axes[0].bar(splits, sizes, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_title('Dataset Split Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples', fontsize=12)
axes[0].set_xlabel('Dataset Type', fontsize=12)
for i, v in enumerate(sizes):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(sizes, labels=splits, autopct='%1.1f%%', colors=colors, 
            startangle=90, explode=(0.05, 0.05, 0.05))
axes[1].set_title('Dataset Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"✓ Total dataset size: {sum(sizes)} samples")

In [ ]:
# Visualize average text length by language
avg_lengths = {lang: train_df[lang].str.len().mean() for lang in languages}

plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(languages)), list(avg_lengths.values()), 
               color=plt.cm.viridis(np.linspace(0, 1, len(languages))),
               edgecolor='black', alpha=0.8)

plt.xlabel('Language', fontsize=12, fontweight='bold')
plt.ylabel('Average Text Length (characters)', fontsize=12, fontweight='bold')
plt.title('Average Text Length by Language', fontsize=14, fontweight='bold')
plt.xticks(range(len(languages)), languages, rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (lang, val) in enumerate(avg_lengths.items()):
    plt.text(i, val + 2, f'{val:.1f}', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete")

In [ ]:
# Distribution of text lengths (histogram)
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for idx, lang in enumerate(languages):
    lengths = train_df[lang].str.len()
    axes[idx].hist(lengths, bins=50, color=plt.cm.tab10(idx), alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'{lang}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Text Length', fontsize=9)
    axes[idx].set_ylabel('Frequency', fontsize=9)
    axes[idx].axvline(lengths.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {lengths.mean():.1f}')
    axes[idx].legend(fontsize=8)
    axes[idx].grid(alpha=0.3)

plt.suptitle('Text Length Distribution Across Languages', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("✓ Distribution analysis complete")

## Step 4: Feature Engineering & Text Preprocessing

In [ ]:
# Import preprocessing libraries
import re
import unicodedata

print("✓ Preprocessing libraries imported")

# Define special tokens
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
SOS_TOKEN = '<SOS>'  # Start of sequence
EOS_TOKEN = '<EOS>'  # End of sequence

print(f"✓ Special tokens defined: {PAD_TOKEN}, {UNK_TOKEN}, {SOS_TOKEN}, {EOS_TOKEN}")

In [ ]:
# Text preprocessing functions
def normalize_text(text):
    """Normalize text by removing accents and converting to lowercase"""
    text = text.lower().strip()
    # Normalize unicode characters
    text = unicodedata.normalize('NFD', text)
    text = ''.join(char for char in text if unicodedata.category(char) != 'Mn')
    # Add spaces around punctuation
    text = re.sub(r"([.!?])", r" \1", text)
    text = re.sub(r"[^a-zA-Z.!?]+", r" ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    """Simple word tokenization"""
    return text.split()

# Test preprocessing
sample_text = train_df['en_US'].iloc[0]
print(f"Original: {sample_text}")
print(f"Normalized: {normalize_text(sample_text)}")
print(f"Tokenized: {tokenize(normalize_text(sample_text))}")
print("\n✓ Preprocessing functions defined")

In [ ]:
# Build vocabulary class
class Vocabulary:
    def __init__(self, freq_threshold=2):
        self.itos = {0: PAD_TOKEN, 1: SOS_TOKEN, 2: EOS_TOKEN, 3: UNK_TOKEN}
        self.stoi = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2, UNK_TOKEN: 3}
        self.freq_threshold = freq_threshold
        
    def __len__(self):
        return len(self.itos)
    
    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4
        
        for sentence in sentence_list:
            for word in tokenize(sentence):
                frequencies[word] += 1
                
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1
    
    def numericalize(self, text):
        tokenized_text = tokenize(text)
        return [self.stoi.get(token, self.stoi[UNK_TOKEN]) for token in tokenized_text]

print("✓ Vocabulary class defined")

## Step 5: Build Vocabularies for All Languages

In [ ]:
# Build vocabularies for all languages
print("Building vocabularies for all languages...")
print("=" * 60)

vocabularies = {}
for lang in languages:
    print(f"Building vocabulary for {lang}...", end=" ")
    vocab = Vocabulary(freq_threshold=2)
    
    # Normalize all texts first
    normalized_texts = [normalize_text(str(text)) for text in train_df[lang]]
    vocab.build_vocabulary(normalized_texts)
    
    vocabularies[lang] = vocab
    print(f"✓ Size: {len(vocab)} tokens")

print("=" * 60)
print(f"✓ All vocabularies built successfully!")
print(f"Total vocabularies: {len(vocabularies)}")

## Step 6: Create Dataset and DataLoader (TensorFlow)

In [ ]:
# Prepare data for TensorFlow
def prepare_sequences(dataframe, src_lang, tgt_lang, src_vocab, tgt_vocab, max_len=50):
    src_sequences = []
    tgt_sequences = []
    
    for idx in range(len(dataframe)):
        src_text = normalize_text(str(dataframe.iloc[idx][src_lang]))
        tgt_text = normalize_text(str(dataframe.iloc[idx][tgt_lang]))
        
        # Numericalize source
        src_numericalized = [src_vocab.stoi[SOS_TOKEN]]
        src_numericalized += src_vocab.numericalize(src_text)
        src_numericalized.append(src_vocab.stoi[EOS_TOKEN])
        
        # Numericalize target
        tgt_numericalized = [tgt_vocab.stoi[SOS_TOKEN]]
        tgt_numericalized += tgt_vocab.numericalize(tgt_text)
        tgt_numericalized.append(tgt_vocab.stoi[EOS_TOKEN])
        
        src_sequences.append(src_numericalized)
        tgt_sequences.append(tgt_numericalized)
    
    # Pad sequences
    src_padded = pad_sequences(src_sequences, maxlen=max_len, padding='post', value=0)
    tgt_padded = pad_sequences(tgt_sequences, maxlen=max_len, padding='post', value=0)
    
    return src_padded, tgt_padded

print("✓ Data preparation function defined")

In [ ]:
# Create TensorFlow dataset
def create_tf_dataset(src_data, tgt_data, batch_size=32):
    """Create TensorFlow dataset"""
    dataset = tf.data.Dataset.from_tensor_slices((src_data, tgt_data))
    dataset = dataset.shuffle(buffer_size=len(src_data))
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

print("✓ TensorFlow dataset creation function defined")

## Step 7: Build Encoder-Decoder Model with Attention (Keras)

In [ ]:
# Encoder model
def build_encoder(vocab_size, embedding_size, hidden_size):
    """Build encoder using Keras Functional API"""
    inputs = keras.Input(shape=(None,), dtype=tf.int32)
    
    # Embedding layer
    x = layers.Embedding(input_dim=vocab_size, output_dim=embedding_size)(inputs)
    
    # LSTM layers
    x, state_h, state_c = layers.LSTM(hidden_size, return_state=True, return_sequences=True)(x)
    x, state_h_2, state_c_2 = layers.LSTM(hidden_size, return_state=True)(x)
    
    # States for decoder initialization
    encoder_states = [state_h_2, state_c_2]
    
    encoder = keras.Model(inputs, encoder_states, name='encoder')
    return encoder

print("✓ Encoder model function defined")

In [ ]:
# Decoder model
def build_decoder(vocab_size, embedding_size, hidden_size):
    """Build decoder using Keras Functional API"""
    inputs = keras.Input(shape=(None,), dtype=tf.int32)
    state_h_input = keras.Input(shape=(hidden_size,), dtype=tf.float32)
    state_c_input = keras.Input(shape=(hidden_size,), dtype=tf.float32)
    
    # Embedding layer
    x = layers.Embedding(input_dim=vocab_size, output_dim=embedding_size)(inputs)
    
    # LSTM layers
    x, state_h, state_c = layers.LSTM(hidden_size, return_sequences=True, return_state=True)
                                      (x, initial_state=[state_h_input, state_c_input])
    
    # Output layer
    outputs = layers.Dense(vocab_size, activation='softmax')(x)
    
    decoder = keras.Model([inputs, state_h_input, state_c_input], [outputs, state_h, state_c], name='decoder')
    return decoder

print("✓ Decoder model function defined")

## Step 8: Training Setup and Model Training

In [ ]:
# Hyperparameters
EMBEDDING_SIZE = 256
HIDDEN_SIZE = 512
DROPOUT = 0.5
LEARNING_RATE = 0.001
BATCH_SIZE = 32
NUM_EPOCHS = 10
MAX_SEQ_LEN = 50

print(f"✓ Hyperparameters set:")
print(f"  - Embedding size: {EMBEDDING_SIZE}")
print(f"  - Hidden size: {HIDDEN_SIZE}")
print(f"  - Dropout: {DROPOUT}")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Epochs: {NUM_EPOCHS}")
print(f"  - Max sequence length: {MAX_SEQ_LEN}")

In [ ]:
# Select language pair for training
SRC_LANG = 'en_US'
TGT_LANG = 'es_ES'

print(f"Training model for: {SRC_LANG} → {TGT_LANG}")
print("=" * 60)

# Get vocabularies
src_vocab = vocabularies[SRC_LANG]
tgt_vocab = vocabularies[TGT_LANG]

print(f"Source vocab size: {len(src_vocab)}")
print(f"Target vocab size: {len(tgt_vocab)}")

# Prepare sequences
print("\nPreparing training sequences...")
src_train, tgt_train = prepare_sequences(train_df, SRC_LANG, TGT_LANG, src_vocab, tgt_vocab, max_len=MAX_SEQ_LEN)
print(f"Source sequences shape: {src_train.shape}")
print(f"Target sequences shape: {tgt_train.shape}")

# Create TensorFlow dataset
train_dataset = create_tf_dataset(src_train, tgt_train, batch_size=BATCH_SIZE)
print(f"Number of training batches: {len(train_dataset)}")
print(f"✓ Dataset prepared")

In [ ]:
# Build Seq2Seq model with Encoder-Decoder
print("Building Seq2Seq model...")

# Build encoder and decoder
encoder = build_encoder(len(src_vocab), EMBEDDING_SIZE, HIDDEN_SIZE)
decoder = build_decoder(len(tgt_vocab), EMBEDDING_SIZE, HIDDEN_SIZE)

# Build full seq2seq model
encoder_inputs = keras.Input(shape=(None,), dtype=tf.int32)
decoder_inputs = keras.Input(shape=(None,), dtype=tf.int32)

# Encoder
encoder_states = encoder(encoder_inputs)

# Decoder (with encoder states as initial state)
decoder_outputs, _, _ = decoder([decoder_inputs, encoder_states[0], encoder_states[1]])

# Model
seq2seq_model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs, name='seq2seq')

# Compile model
seq2seq_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

seq2seq_model.summary()
print("\n✓ Model built and compiled")

In [ ]:
# Custom callback for progress tracking
class CustomCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS} - Loss: {logs['loss']:.4f}, Accuracy: {logs['accuracy']:.4f}")

print("✓ Custom callback defined")

In [ ]:
# Train model
print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60 + "\n")

history = seq2seq_model.fit(
    [src_train, tgt_train[:, :-1]], tgt_train[:, 1:].reshape(tgt_train.shape[0], MAX_SEQ_LEN - 1, 1),
    batch_size=BATCH_SIZE,
    epochs=NUM_EPOCHS,
    validation_split=0.2,
    callbacks=[CustomCallback()],
    verbose=0
)

print("\n" + "=" * 60)
print("✓ Training completed!")
print("=" * 60)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2, marker='o', markersize=5)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2, marker='s', markersize=5)
axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Loss', fontsize=12, fontweight='bold')
axes[0].set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2, marker='o', markersize=5)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2, marker='s', markersize=5)
axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[1].set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Training visualization complete")

## Step 9: Save Model (HDF5 Format)

In [ ]:
# Create models directory
os.makedirs('models', exist_ok=True)

# Save model
model_save_path = f'models/translator_{SRC_LANG}_to_{TGT_LANG}_tf.h5'
seq2seq_model.save(model_save_path)

# Save vocabularies
vocab_save_path = f'models/vocabularies_{SRC_LANG}_to_{TGT_LANG}.pkl'
vocab_data = {
    'src_vocab': src_vocab,
    'tgt_vocab': tgt_vocab,
    'src_lang': SRC_LANG,
    'tgt_lang': TGT_LANG,
    'hyperparameters': {
        'embedding_size': EMBEDDING_SIZE,
        'hidden_size': HIDDEN_SIZE,
        'max_seq_len': MAX_SEQ_LEN
    }
}

with open(vocab_save_path, 'wb') as f:
    pickle.dump(vocab_data, f)

print(f"✓ Model saved to: {model_save_path}")
print(f"✓ Model size: {os.path.getsize(model_save_path) / 1024 / 1024:.2f} MB")
print(f"✓ Vocabularies saved to: {vocab_save_path}")

## Step 10: Model Testing & Translation Function

In [ ]:
# Inference model for translation
def build_inference_encoder():
    """Build encoder for inference"""
    return encoder

# Prepare encoder and decoder for inference
inference_encoder = build_inference_encoder()

print("✓ Inference models ready")

In [ ]:
# Translation function
def translate_sentence(sentence, src_vocab, tgt_vocab, encoder_model, decoder_model, max_length=MAX_SEQ_LEN):
    """Translate a sentence"""
    # Preprocess input
    sentence = normalize_text(sentence)
    tokens = [src_vocab.stoi[SOS_TOKEN]] + src_vocab.numericalize(sentence) + [src_vocab.stoi[EOS_TOKEN]]
    
    # Pad to max_length
    src_tensor = np.array([tokens])
    src_tensor = pad_sequences([tokens], maxlen=max_length, padding='post', value=0)
    
    # Get encoder states
    encoder_states = encoder_model.predict(src_tensor, verbose=0)
    
    # Start with SOS token
    target_seq = np.zeros((1, max_length))
    target_seq[0, 0] = tgt_vocab.stoi[SOS_TOKEN]
    
    # Decode step by step
    translated_tokens = [tgt_vocab.stoi[SOS_TOKEN]]
    
    for i in range(1, max_length):
        target_seq = np.zeros((1, max_length))
        target_seq[0, :i] = translated_tokens
        
        output, state_h, state_c = decoder_model.predict([target_seq, encoder_states[0], encoder_states[1]], verbose=0)
        
        predicted_id = np.argmax(output[0, i-1, :])
        translated_tokens.append(predicted_id)
        
        if predicted_id == tgt_vocab.stoi[EOS_TOKEN]:
            break
    
    # Convert to words
    translated_text = ' '.join([tgt_vocab.itos[idx] for idx in translated_tokens if idx in tgt_vocab.itos])
    return translated_text.replace(SOS_TOKEN, '').replace(EOS_TOKEN, '').strip()

print("✓ Translation function defined")

In [ ]:
# Test the model with sample translations
print("=" * 60)
print("TESTING TRANSLATIONS")
print("=" * 60)

test_sentences = [
    "Hello, how are you?",
    "I love programming.",
    "The weather is beautiful today.",
    "What time is it?",
    "Thank you very much."
]

for idx, sentence in enumerate(test_sentences, 1):
    translation = translate_sentence(sentence, src_vocab, tgt_vocab, encoder, decoder, max_length=MAX_SEQ_LEN)
    
    print(f"\n{idx}. Source ({SRC_LANG}): {sentence}")
    print(f"   Predicted ({TGT_LANG}): {translation}")
    print("-" * 60)

print("\n✓ Testing complete!")